In [1]:
import sys

sys.path.append("..")

from common.metrics import metrics
from common.preprocessing import preprocessing_cn7
from common.train_test_split import split_data
from common.model_composition import model_composition

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 데이터 불러오기
train_cn7 = pd.read_csv(r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_cn7.csv")

In [3]:
X_cn7, y_cn7 = preprocessing_cn7(train_cn7)

print("X shape:", X_cn7.shape)
print("y shape:", y_cn7.shape)

X shape: (1211, 23)
y shape: (1211,)


### baseline 교차검증

In [4]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        random_state=42,
        verbose=-1
    ),

    "XGBoost": XGBClassifier(
        random_state=42
    ),

    "CatBoost": CatBoostClassifier(
        random_state=42,
        verbose=0
    )
}

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42
)

scoring = {
    "PR-AUC": "average_precision",
    "Recall": "recall",
    "Precision": "precision",
    "F1": "f1"
}

In [5]:
import pandas as pd

results = []

for name, model in models.items():

    scores = cross_validate(
        model,
        X_cn7,
        y_cn7,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    results.append({
        "Model": name,

        "PR-AUC Mean": scores["test_PR-AUC"].mean(),
        "PR-AUC Std": scores["test_PR-AUC"].std(),

        "Recall Mean": scores["test_Recall"].mean(),
        "Recall Std": scores["test_Recall"].std(),

        "Precision Mean": scores["test_Precision"].mean(),
        "Precision Std": scores["test_Precision"].std(),

        "F1 Mean": scores["test_F1"].mean(),
        "F1 Std": scores["test_F1"].std()
    })

cv_baseline = pd.DataFrame(results)

display(
    cv_baseline.sort_values(
        "PR-AUC Mean",
        ascending=False
    )
)

,Model,PR-AUC Mean,PR-AUC Std,Recall Mean,Recall Std,Precision Mean,Precision Std,F1 Mean,F1 Std
0,Logistic Regression,0.497936,0.220785,0.340000,0.233238,0.740000,0.427083,0.455619,0.287246
2,LightGBM,0.462618,0.188005,0.340000,0.233238,0.386857,0.287991,0.336000,0.214165
4,CatBoost,0.451184,0.195733,0.340000,0.233238,0.386857,0.287991,0.336000,0.214165
3,XGBoost,0.420934,0.152535,0.293333,0.214890,0.364762,0.293811,0.304762,0.212063
1,Random Forest,0.401487,0.214712,0.340000,0.233238,0.386857,0.287991,0.336000,0.214165


### 5fold + class weight

In [6]:
models_cw = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        class_weight="balanced",
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        class_weight="balanced",
        random_state=42,
        verbose=-1
    ),

    "XGBoost": XGBClassifier(
        scale_pos_weight=(y_cn7 == 0).sum() / (y_cn7 == 1).sum(),
        random_state=42
    ),

    "CatBoost": CatBoostClassifier(
        auto_class_weights="Balanced",
        random_state=42,
        verbose=0
    )
}

In [7]:
results_cw = []

for name, model in models_cw.items():

    scores = cross_validate(
        model,
        X_cn7,
        y_cn7,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    results_cw.append({
        "Model": name,

        "PR-AUC Mean": scores["test_PR-AUC"].mean(),
        "PR-AUC Std": scores["test_PR-AUC"].std(),

        "Recall Mean": scores["test_Recall"].mean(),
        "Recall Std": scores["test_Recall"].std(),

        "Precision Mean": scores["test_Precision"].mean(),
        "Precision Std": scores["test_Precision"].std(),

        "F1 Mean": scores["test_F1"].mean(),
        "F1 Std": scores["test_F1"].std()
    })

cv_class_weight = pd.DataFrame(results_cw)

display(
    cv_class_weight.sort_values(
        "PR-AUC Mean",
        ascending=False
    )
)

,Model,PR-AUC Mean,PR-AUC Std,Recall Mean,Recall Std,Precision Mean,Precision Std,F1 Mean,F1 Std
0,Logistic Regression,0.486761,0.235565,0.613333,0.253728,0.154477,0.072752,0.243049,0.107864
2,LightGBM,0.470160,0.179102,0.340000,0.233238,0.354857,0.261292,0.328095,0.214563
4,CatBoost,0.446971,0.191037,0.340000,0.233238,0.371524,0.290212,0.329333,0.215438
3,XGBoost,0.371410,0.165888,0.350000,0.254951,0.353714,0.260278,0.330519,0.218677
1,Random Forest,0.276477,0.111284,0.340000,0.233238,0.358857,0.287435,0.322571,0.211478


### CV + Randomsampling

In [8]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler

models_ros = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        random_state=42,
        verbose=-1
    ),

    "XGBoost": XGBClassifier(
        random_state=42
    ),

    "CatBoost": CatBoostClassifier(
        random_state=42,
        verbose=0
    )
}

In [9]:
results_ros = []

for name, model in models_ros.items():

    pipeline = Pipeline([
        ("ros", RandomOverSampler(random_state=42)),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        X_cn7,
        y_cn7,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    results_ros.append({
        "Model": name,

        "PR-AUC Mean": scores["test_PR-AUC"].mean(),
        "PR-AUC Std": scores["test_PR-AUC"].std(),

        "Recall Mean": scores["test_Recall"].mean(),
        "Recall Std": scores["test_Recall"].std(),

        "Precision Mean": scores["test_Precision"].mean(),
        "Precision Std": scores["test_Precision"].std(),

        "F1 Mean": scores["test_F1"].mean(),
        "F1 Std": scores["test_F1"].std()
    })

cv_ros = pd.DataFrame(results_ros)

display(
    cv_ros.sort_values(
        "PR-AUC Mean",
        ascending=False
    )
)

,Model,PR-AUC Mean,PR-AUC Std,Recall Mean,Recall Std,Precision Mean,Precision Std,F1 Mean,F1 Std
0,Logistic Regression,0.495934,0.239126,0.603333,0.253070,0.167641,0.080268,0.256669,0.113006
4,CatBoost,0.436218,0.194772,0.340000,0.233238,0.371524,0.290212,0.329333,0.215438
2,LightGBM,0.426772,0.153599,0.340000,0.233238,0.344857,0.260328,0.324762,0.215102
3,XGBoost,0.387403,0.174415,0.350000,0.244949,0.381524,0.290159,0.336667,0.218801
1,Random Forest,0.273647,0.120991,0.320000,0.218174,0.370857,0.286814,0.317429,0.203826


### 5-fold + smote

In [10]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

results_smote = []

for name, model in models_ros.items():

    pipeline = Pipeline([
        ("smote", SMOTE(
            random_state=42,
            k_neighbors=3
        )),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        X_cn7,
        y_cn7,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    results_smote.append({
        "Model": name,

        "PR-AUC Mean": scores["test_PR-AUC"].mean(),
        "PR-AUC Std": scores["test_PR-AUC"].std(),

        "Recall Mean": scores["test_Recall"].mean(),
        "Recall Std": scores["test_Recall"].std(),

        "Precision Mean": scores["test_Precision"].mean(),
        "Precision Std": scores["test_Precision"].std(),

        "F1 Mean": scores["test_F1"].mean(),
        "F1 Std": scores["test_F1"].std()
    })

cv_smote = pd.DataFrame(results_smote)

display(
    cv_smote.sort_values(
        "PR-AUC Mean",
        ascending=False
    )
)

,Model,PR-AUC Mean,PR-AUC Std,Recall Mean,Recall Std,Precision Mean,Precision Std,F1 Mean,F1 Std
0,Logistic Regression,0.488262,0.231439,0.593333,0.252014,0.173431,0.079816,0.264407,0.115145
2,LightGBM,0.446615,0.170474,0.340000,0.233238,0.371524,0.290212,0.326000,0.212101
1,Random Forest,0.410735,0.180587,0.340000,0.233238,0.371524,0.290212,0.326000,0.212101
3,XGBoost,0.403997,0.180279,0.350000,0.234521,0.350857,0.283327,0.320921,0.207636
4,CatBoost,0.338866,0.161862,0.350000,0.234521,0.371524,0.290212,0.332667,0.214828
